In [2]:
"""
PROJECT 1 - Naive Bayes Spam Classifier from Scratch
"""
import math
import re
from collections import defaultdict

In [3]:
#-----------------------------------------------
#1. TRAINING DATA - 20 hardcoded emails
#-----------------------------------------------
emails = [
    #SPAM (label=1)
    ("win free money now click here prize lottery",1),
    ("congratulations you won a free iphone claim now",1),
    ("cheap viagra buy now limited offer discount",1),
    ("urgent your account needs verification click link",1),
    ("earn money fast work from home guaranteed income",1),
    ("free offer win cash prize today only hurry",1),
    ("click here to claim your reward free gift now",1),
    ("hot singles in your area meet them tonight free",1),
    ("you have been selected for a special cash bonus",1),
    ("limited time offer buy now and save big discount",1),
    #HAM(label=0)
    ("hey are you free for lunch tomorrow",0),
    ("the project report is due on friday please review",0),
    ("can you pick up milk on your way home tonight",0),
    ("meeting rescheduled to 3pm let me know if that works",0),
    ("happy birthday hope you have a wonderful day",0),
    ("attached is the invoice for last months services",0),
    ("reminder your dentist appointment is on thursday",0),
    ("just checking in how was your weekend trip",0),
    ("the quarterly results are ready for your review",0),
    ("could you send me the updated project timeline",0)
]


In [4]:
#------------------------------------------------------------------
#2. TEXT PREPROCESSING
#------------------------------------------------------------------
def tokenize(text):
    """Lowercase and split into words, removing non-aplha characters."""
    return re.findall(r'[a-z]+',text.lower())

In [10]:
class NaiveBayesClassifier:
    """
    Multinomial Naive Bayes with Laplace (add-1) smoothing.

    Math recap:
        P(spam | words) ∝ P(spam) × ∏ P(word | spam)

    With Laplace smoothing:
        P(word | class) = (count(word, class) + 1) / (total_words_in_class + vocab_size)
    """

    def __init__(self):
        self.class_word_counts = defaultdict(lambda: defaultdict(int))
        self.class_totals      = defaultdict(int)   # total word count per class
        self.class_doc_counts  = defaultdict(int)   # number of docs per class
        self.vocab             = set()
        self.total_docs        = 0

    # ── TRAINING ──────────────────────────────
    def fit(self, training_data):
        """
        training_data: list of (text_string, label) tuples
        """
        for text, label in training_data:
            words = tokenize(text)
            self.class_doc_counts[label] += 1
            self.total_docs += 1
            for word in words:
                self.class_word_counts[label][word] += 1
                self.class_totals[label] += 1
                self.vocab.add(word)

        self.vocab_size = len(self.vocab)
        self.classes    = list(self.class_doc_counts.keys())

        # Prior probabilities: P(class) = docs_in_class / total_docs
        self.log_priors = {
            c: math.log(self.class_doc_counts[c] / self.total_docs)
            for c in self.classes
        }

    # ── LIKELIHOOD (with Laplace smoothing) ───
    def log_likelihood(self, word, cls):
        """
        log P(word | class) using Laplace smoothing.
        Uses log to avoid floating-point underflow from multiplying tiny probabilities.
        """
        count = self.class_word_counts[cls].get(word, 0)
        return math.log((count + 1) / (self.class_totals[cls] + self.vocab_size))

    # ── PREDICTION ────────────────────────────
    def predict(self, text):
        """Return predicted class label for a single text string."""
        words = tokenize(text)
        scores = {}
        for cls in self.classes:
            # Start with log prior, add log likelihoods
            score = self.log_priors[cls]
            for word in words:
                score += self.log_likelihood(word, cls)
            scores[cls] = score
        return max(scores, key=scores.get)

    def predict_proba(self, text):
        """Return normalised probability estimates for each class."""
        words = tokenize(text)
        log_scores = {}
        for cls in self.classes:
            score = self.log_priors[cls]
            for word in words:
                score += self.log_likelihood(word, cls)
            log_scores[cls] = score

        # Convert log-scores to probabilities via softmax
        max_score = max(log_scores.values())
        exp_scores = {c: math.exp(s - max_score) for c, s in log_scores.items()}
        total = sum(exp_scores.values())
        return {c: round(v / total, 4) for c, v in exp_scores.items()}

class NaiveBayesClassifier:
    """
    Multinomial Naive Bayes with Laplace (add-1) smoothing.

    Math recap:
        P(spam | words) ∝ P(spam) × ∏ P(word | spam)

    With Laplace smoothing:
        P(word | class) = (count(word, class) + 1) / (total_words_in_class + vocab_size)
    """

    def __init__(self):
        self.class_word_counts = defaultdict(lambda: defaultdict(int))
        self.class_totals      = defaultdict(int)   # total word count per class
        self.class_doc_counts  = defaultdict(int)   # number of docs per class
        self.vocab             = set()
        self.total_docs        = 0

    # ── TRAINING ──────────────────────────────
    def fit(self, training_data):
        """
        training_data: list of (text_string, label) tuples
        """
        for text, label in training_data:
            words = tokenize(text)
            self.class_doc_counts[label] += 1
            self.total_docs += 1
            for word in words:
                self.class_word_counts[label][word] += 1
                self.class_totals[label] += 1
                self.vocab.add(word)

        self.vocab_size = len(self.vocab)
        self.classes    = list(self.class_doc_counts.keys())

        # Prior probabilities: P(class) = docs_in_class / total_docs
        self.log_priors = {
            c: math.log(self.class_doc_counts[c] / self.total_docs)
            for c in self.classes
        }

    # ── LIKELIHOOD (with Laplace smoothing) ───
    def log_likelihood(self, word, cls):
        """
        log P(word | class) using Laplace smoothing.
        Uses log to avoid floating-point underflow from multiplying tiny probabilities.
        """
        count = self.class_word_counts[cls].get(word, 0)
        return math.log((count + 1) / (self.class_totals[cls] + self.vocab_size))

    # ── PREDICTION ────────────────────────────
    def predict(self, text):
        """Return predicted class label for a single text string."""
        words = tokenize(text)
        scores = {}
        for cls in self.classes:
            # Start with log prior, add log likelihoods
            score = self.log_priors[cls]
            for word in words:
                score += self.log_likelihood(word, cls)
            scores[cls] = score
        return max(scores, key=scores.get)

    def predict_proba(self, text):
        """Return normalised probability estimates for each class."""
        words = tokenize(text)
        log_scores = {}
        for cls in self.classes:
            score = self.log_priors[cls]
            for word in words:
                score += self.log_likelihood(word, cls)
            log_scores[cls] = score

        # Convert log-scores to probabilities via softmax
        max_score = max(log_scores.values())
        exp_scores = {c: math.exp(s - max_score) for c, s in log_scores.items()}
        total = sum(exp_scores.values())
        return {c: round(v / total, 4) for c, v in exp_scores.items()}

class NaiveBayesClassifier:
    """
    Multinomial Naive Bayes with Laplace (add-1) smoothing.

    Math recap:
        P(spam | words) ∝ P(spam) × ∏ P(word | spam)

    With Laplace smoothing:
        P(word | class) = (count(word, class) + 1) / (total_words_in_class + vocab_size)
    """

    def __init__(self):
        self.class_word_counts = defaultdict(lambda: defaultdict(int))
        self.class_totals      = defaultdict(int)   # total word count per class
        self.class_doc_counts  = defaultdict(int)   # number of docs per class
        self.vocab             = set()
        self.total_docs        = 0

    # ── TRAINING ──────────────────────────────
    def fit(self, training_data):
        """
        training_data: list of (text_string, label) tuples
        """
        for text, label in training_data:
            words = tokenize(text)
            self.class_doc_counts[label] += 1
            self.total_docs += 1
            for word in words:
                self.class_word_counts[label][word] += 1
                self.class_totals[label] += 1
                self.vocab.add(word)

        self.vocab_size = len(self.vocab)
        self.classes    = list(self.class_doc_counts.keys())

        # Prior probabilities: P(class) = docs_in_class / total_docs
        self.log_priors = {
            c: math.log(self.class_doc_counts[c] / self.total_docs)
            for c in self.classes
        }

    # ── LIKELIHOOD (with Laplace smoothing) ───
    def log_likelihood(self, word, cls):
        """
        log P(word | class) using Laplace smoothing.
        Uses log to avoid floating-point underflow from multiplying tiny probabilities.
        """
        count = self.class_word_counts[cls].get(word, 0)
        return math.log((count + 1) / (self.class_totals[cls] + self.vocab_size))

    # ── PREDICTION ────────────────────────────
    def predict(self, text):
        """Return predicted class label for a single text string."""
        words = tokenize(text)
        scores = {}
        for cls in self.classes:
            # Start with log prior, add log likelihoods
            score = self.log_priors[cls]
            for word in words:
                score += self.log_likelihood(word, cls)
            scores[cls] = score
        return max(scores, key=scores.get)

    def predict_proba(self, text):
        """Return normalised probability estimates for each class."""
        words = tokenize(text)
        log_scores = {}
        for cls in self.classes:
            score = self.log_priors[cls]
            for word in words:
                score += self.log_likelihood(word, cls)
            log_scores[cls] = score

        # Convert log-scores to probabilities via softmax
        max_score = max(log_scores.values())
        exp_scores = {c: math.exp(s - max_score) for c, s in log_scores.items()}
        total = sum(exp_scores.values())
        return {c: round(v / total, 4) for c, v in exp_scores.items()}

class NaiveBayesClassifier:
    """
    Multinomial Naive Bayes with Laplace (add-1) smoothing.

    Math recap:
        P(spam | words) ∝ P(spam) × ∏ P(word | spam)

    With Laplace smoothing:
        P(word | class) = (count(word, class) + 1) / (total_words_in_class + vocab_size)
    """

    def __init__(self):
        self.class_word_counts = defaultdict(lambda: defaultdict(int))
        self.class_totals      = defaultdict(int)   # total word count per class
        self.class_doc_counts  = defaultdict(int)   # number of docs per class
        self.vocab             = set()
        self.total_docs        = 0

    # ── TRAINING ──────────────────────────────
    def fit(self, training_data):
        """
        training_data: list of (text_string, label) tuples
        """
        for text, label in training_data:
            words = tokenize(text)
            self.class_doc_counts[label] += 1
            self.total_docs += 1
            for word in words:
                self.class_word_counts[label][word] += 1
                self.class_totals[label] += 1
                self.vocab.add(word)

        self.vocab_size = len(self.vocab)
        self.classes    = list(self.class_doc_counts.keys())

        # Prior probabilities: P(class) = docs_in_class / total_docs
        self.log_priors = {
            c: math.log(self.class_doc_counts[c] / self.total_docs)
            for c in self.classes
        }

    # ── LIKELIHOOD (with Laplace smoothing) ───
    def log_likelihood(self, word, cls):
        """
        log P(word | class) using Laplace smoothing.
        Uses log to avoid floating-point underflow from multiplying tiny probabilities.
        """
        count = self.class_word_counts[cls].get(word, 0)
        return math.log((count + 1) / (self.class_totals[cls] + self.vocab_size))

    # ── PREDICTION ────────────────────────────
    def predict(self, text):
        """Return predicted class label for a single text string."""
        words = tokenize(text)
        scores = {}
        for cls in self.classes:
            # Start with log prior, add log likelihoods
            score = self.log_priors[cls]
            for word in words:
                score += self.log_likelihood(word, cls)
            scores[cls] = score
        return max(scores, key=scores.get)

    def predict_proba(self, text):
        """Return normalised probability estimates for each class."""
        words = tokenize(text)
        log_scores = {}
        for cls in self.classes:
            score = self.log_priors[cls]
            for word in words:
                score += self.log_likelihood(word, cls)
            log_scores[cls] = score

        # Convert log-scores to probabilities via softmax
        max_score = max(log_scores.values())
        exp_scores = {c: math.exp(s - max_score) for c, s in log_scores.items()}
        total = sum(exp_scores.values())
        return {c: round(v / total, 4) for c, v in exp_scores.items()}



In [11]:
#-------------------------------------------
##4 Word Frequency Counter
#-------------------------------------------
def word_frequency_report(classifier):
    print("\n--TOP 10 WORDS PER CLASS------------------")
    for cls in  [0,1]:
        label = "SPAM" if cls ==1 else "HAM"
        sorted_words = sorted(
            classifier.class_word_counts[cls].items(),
            key=lambda x: x[1], reverse = True
        )[:10]
        print(f"{label}:{sorted_words}")

In [12]:
#--------------------------------------------------------
#5. SKLEARN COMPARISON
#--------------------------------------------------------
def sklearn_comparison(train_texts, train_labels, test_texts, test_labels):
    try:
        from sklearn.naive_bayes import MultinomialNB
        from sklearn.feature_extraction.text import CountVectorizer

        vec = CountVectrizer()
        X_tr = vec.fit_transform(train_texts)
        X_te = vec.transform(test_texts)
        model = MultinomialNB()
        model.fit(X_tr, train_labels)
        preds = model.predict(X_te)

        correct = sum(p == a for p,a  in zip(preds, test_labels))
        acc = correct/len(test_labels)
        print(f"\n-----------SKLEARN MultinomialNB-------------")
        print(f" Predictions: {list(preds)}")
        print(f" Accuracy: {acc:.0%}({correct}/{len(test_labels)})")
        return acc
    except ImportError:
        print("Sklearn not installed -Skipping comparison")
        return None
        

In [ ]:
#------------------------------------------------------
#6. MAIN - TRAIN, TEST, COMPARE
#------------------------------------------------------
if __name__ == "__main__":

    # ── Train ─────────────────────────────────
    clf = NaiveBayesClassifier()
    clf.fit(emails)

    print("=" * 55)
    print("  NAIVE BAYES SPAM CLASSIFIER  (from scratch)")
    print("=" * 55)
    print(f"  Training docs : {clf.total_docs}")
    print(f"  Vocabulary    : {clf.vocab_size} unique words")
    print(f"  Log priors    → SPAM: {clf.log_priors[1]:.4f} | HAM: {clf.log_priors[0]:.4f}")
    print(f"  Prior P(spam) : {math.exp(clf.log_priors[1]):.2f}  "
          f"| P(ham): {math.exp(clf.log_priors[0]):.2f}")

    word_frequency_report(clf)
    test_emails = [
        ("win cash prize free lottery click now", 1),          # spam
        ("can we schedule a call for tomorrow afternoon", 0),  # ham
        ("buy cheap pills online no prescription needed", 1),  # spam
        ("please find attached the monthly budget report", 0), # ham
        ("you are selected for exclusive reward claim today", 1), # spam
    ]

    print("\n── TEST PREDICTIONS ────────────────────────────")
    print(f"  {'Email (truncated)':<45} {'True':>5} {'Pred':>5} {'P(spam)':>8}")
    print("  " + "-" * 68)

    scratch_correct = 0
    for text, true_label in test_emails:
        pred  = clf.predict(text)
        proba = clf.predict_proba(text)
        match = "✓" if pred == true_label else "✗"
        scratch_correct += int(pred == true_label)
        snippet = text[:43] + ".." if len(text) > 45 else text
        print(f"  {snippet:<45} {true_label:>5} {pred:>5}  {proba[1]:>7.1%}  {match}")

    scratch_acc = scratch_correct / len(test_emails)
    print(f"\n  Scratch accuracy : {scratch_acc:.0%}  ({scratch_correct}/{len(test_emails)})")

    # ── sklearn comparison ────────────────────
    train_texts  = [t for t, _ in emails]
    train_labels = [l for _, l in emails]
    test_texts   = [t for t, _ in test_emails]
    test_labels  = [l for _, l in test_emails]

    sklearn_acc = sklearn_comparison(train_texts, train_labels, test_texts, test_labels)

    if sklearn_acc is not None:
        print(f"\n── ACCURACY COMPARISON ────────────────────────")
        print(f"  From-scratch NB : {scratch_acc:.0%}")
        print(f"  sklearn NB      : {sklearn_acc:.0%}")
        delta = scratch_acc - sklearn_acc
        print(f"  Difference      : {delta:+.0%}")

    print("\n── HOW LAPLACE SMOOTHING WORKS ─────────────────")
    print("  Without smoothing, any unseen word gives P=0,")
    print("  making the entire product = 0 (zero-frequency problem).")
    print("  Laplace adds +1 to every word count so no probability")
    print("  is ever zero, even for words not seen in training.")
    print("  Formula: P(w|c) = (count(w,c) + 1) / (total_c + |V|)")
    print("=" * 55)

  NAIVE BAYES SPAM CLASSIFIER  (from scratch)
  Training docs : 20
  Vocabulary    : 111 unique words
  Log priors    → SPAM: -0.6931 | HAM: -0.6931
  Prior P(spam) : 0.50  | P(ham): 0.50

--TOP 10 WORDS PER CLASS------------------
HAM:[('you', 4), ('the', 4), ('your', 4), ('for', 3), ('is', 3), ('on', 3), ('are', 2), ('project', 2), ('review', 2), ('me', 2)]
SPAM:[('free', 5), ('now', 5), ('click', 3), ('offer', 3), ('your', 3), ('win', 2), ('money', 2), ('here', 2), ('prize', 2), ('you', 2)]

── TEST PREDICTIONS ────────────────────────────
  Email (truncated)                              True  Pred  P(spam)
  --------------------------------------------------------------------
  win cash prize free lottery click now             1     1   100.0%  ✓
  can we schedule a call for tomorrow afternoon     0     0    16.4%  ✓
  buy cheap pills online no prescription needed     1     1    86.2%  ✓
  please find attached the monthly budget rep..     0     0     2.5%  ✓
  you are selected for 